# Week 2 · SQL with DuckDB: joins, aggregations, window functions

# Requirements: pip install duckdb pandas numpy
# The first code cell installs duckdb via `%pip` (works in VS Code, Jupyter, and Databricks).

This notebook asks the *analytics* questions of the cleaned shipment data in SQL:
on-time rate by carrier, lane, and month; a **window function** ranking carriers within
each month; and a **top-lane** analysis. DuckDB is an embedded analytical database, it
runs SQL directly over your tables in-process, no server, no install beyond the pip.

In [ ]:
%pip install duckdb
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from zoro import data

print("duckdb:", duckdb.__version__)

### Why SQL, and why DuckDB

Python/pandas is how you *shape* a table; SQL is how you *ask questions* of it, and
how you will talk to DuckDB now, Spark later, and the Databricks lakehouse in Weeks
21 to 24. DuckDB is the perfect bridge: it reads the same DataFrames (or CSVs) you already
have, executes standard SQL in-process, and requires zero infrastructure. The three
verbs that matter this week are **join**, **aggregate**, and **window**.

In [ ]:
repo = pathlib.Path.cwd()
while not (repo / "zoro").is_dir() and repo != repo.parent:
    repo = repo.parent

silver = repo / "data" / "silver"
if (silver / "shipments_silver.csv").exists():
    carriers = pd.read_csv(silver / "carriers_silver.csv")
    lanes = pd.read_csv(silver / "lanes_silver.csv")
    ships = pd.read_csv(silver / "shipments_silver.csv", parse_dates=["planned_departure", "planned_arrival", "actual_arrival"])
    print("Loaded silver tables from data/silver/")
else:
    # Rebuild the silver tables deterministically (mirrors 01-pandas-cleaning.ipynb).
    carriers = data.carriers(20, seed=7)
    lanes = data.lanes(20, seed=11)
    ships = data.shipments(100_000, seed=42).drop_duplicates().reset_index(drop=True)
    ships["weight_kg"] = ships["weight_kg"].fillna(ships.groupby("commodity")["weight_kg"].transform("median"))
    lanes = lanes.copy()
    lanes["distance_km"] = lanes["distance_km"].fillna(lanes["distance_km"].median())
    print("Silver not found - regenerated and cleaned in-memory (seed 42).")

con = duckdb.connect()
con.register("carriers", carriers)
con.register("lanes", lanes)
con.register("shipments", ships)
print("Registered carriers, lanes, shipments with DuckDB.")
print("ships:", ships.shape[0], "| lanes:", lanes.shape[0], "| carriers:", carriers.shape[0])

### The join: answer questions across tables

The shipments table stores *foreign keys* (`carrier_id`, `lane_id`), not names. A
**join** re-attaches the master data. Start with the simplest aggregate of all: the
company-wide on-time rate.

In [ ]:
q = "SELECT ROUND(AVG(CAST(is_on_time AS INT)), 4) AS on_time_rate, COUNT(*) AS n FROM shipments"
print(con.execute(q).df().to_string(index=False))

### On-time rate by carrier, by lane, by month

`GROUP BY` collapses rows into groups; the aggregate runs inside each group. The
on-time rate is `AVG` of `is_on_time` cast to 0/1, a rate is just a mean of a
boolean.

In [ ]:
q = '''
SELECT c.carrier_name,
       COUNT(*) AS n_shipments,
       ROUND(AVG(CAST(s.is_on_time AS INT)), 4) AS on_time_rate
FROM shipments s
JOIN carriers c ON s.carrier_id = c.carrier_id
GROUP BY c.carrier_name
ORDER BY on_time_rate DESC
LIMIT 5
'''
print("Top 5 carriers by on-time rate:")
print(con.execute(q).df().to_string(index=False))

In [ ]:
q = '''
SELECT l.lane_id, l.origin, l.destination,
       COUNT(*) AS n_shipments,
       ROUND(AVG(CAST(s.is_on_time AS INT)), 4) AS on_time_rate
FROM shipments s
JOIN lanes l ON s.lane_id = l.lane_id
GROUP BY 1, 2, 3
ORDER BY on_time_rate ASC
LIMIT 5
'''
print("Bottom 5 lanes by on-time rate:")
print(con.execute(q).df().to_string(index=False))

In [ ]:
q = '''
SELECT date_trunc('month', s.planned_departure)::DATE AS month,
       COUNT(*) AS n_shipments,
       ROUND(AVG(CAST(s.is_on_time AS INT)), 4) AS on_time_rate
FROM shipments s
GROUP BY 1
ORDER BY month
'''
print("On-time rate by month:")
print(con.execute(q).df().to_string(index=False))

### Window functions: a value computed across a sliding group

A `GROUP BY` *collapses* rows; a **window function** computes a value across a group
while keeping every row. `RANK() OVER (PARTITION BY ... ORDER BY ...)` ranks each
carrier *within* its own month, and DuckDB's `QUALIFY` filters on the result of a
window function directly, so "top 2 carriers per month" is one clause.

In [ ]:
q = '''
WITH monthly AS (
    SELECT c.carrier_name,
           date_trunc('month', s.planned_departure)::DATE AS month,
           AVG(CAST(s.is_on_time AS INT)) AS on_time_rate
    FROM shipments s
    JOIN carriers c ON s.carrier_id = c.carrier_id
    GROUP BY 1, 2
)
SELECT carrier_name, month, ROUND(on_time_rate, 4) AS on_time_rate,
       RANK() OVER (PARTITION BY month ORDER BY on_time_rate DESC) AS rnk
FROM monthly
QUALIFY rnk <= 2
ORDER BY month, rnk
LIMIT 12
'''
print("Top 2 carriers per month (window function + QUALIFY):")
print(con.execute(q).df().to_string(index=False))

### Top-lane analysis

Which lanes move the most freight, and which are the most profitable? A lane is a
route; aggregate shipment count, average delay, and total value per lane, the kind of
one-query answer an operations analyst wants.

In [ ]:
q = '''
SELECT l.lane_id, l.origin, l.destination,
       COUNT(*) AS shipments,
       ROUND(AVG(s.delay_hours), 2) AS avg_delay_hours,
       ROUND(SUM(s.value_usd), 0) AS revenue_usd
FROM shipments s
JOIN lanes l ON s.lane_id = l.lane_id
GROUP BY 1, 2, 3
ORDER BY shipments DESC
LIMIT 5
'''
print("Top 5 lanes by shipment volume:")
print(con.execute(q).df().to_string(index=False))

### The metric

One number to close: the company-wide on-time rate, as a fraction. It is the headline
KPI every later model either predicts or improves.

In [ ]:
overall_ot = con.execute("SELECT ROUND(AVG(CAST(is_on_time AS INT)), 4) FROM shipments").fetchone()[0]
print("OVERALL_ON_TIME_RATE:", overall_ot)